# Module 7: File Operations for Bioinformatics

## Reading, Writing, and Parsing Biological Data Formats

---

### Learning Objectives

By the end of this module, you will be able to:
- Read and write text files using context managers
- Understand text vs. binary file modes
- Parse FASTA files from scratch (both reader and writer)
- Work with CSV, JSON, and Pickle formats
- Understand GenBank format basics
- Process large files efficiently (line-by-line, generators)
- Handle file-related exceptions properly

---

> *According to informal estimates, up to 90% of a bioinformatician's work involves parsing files and converting between formats!*

File handling is fundamental in bioinformatics. We constantly work with FASTA (sequences), FASTQ (sequencing reads), GenBank (annotated sequences), CSV/TSV (data tables), PDB (protein structures), and many other formats.

## Why this notebook matters

Virtually all bioinformatics data lives in files: FASTA sequences, FASTQ reads, BED intervals, VCF variants, GFF annotations, TSV expression matrices. Reading and writing these files correctly is a prerequisite for every real analysis. This notebook covers the Python `open()` function and `with` statement, then works through the biological file formats you will encounter most often.

## How to use this notebook

1. Run Section 1 first — it creates demo files in the current directory that the rest of the notebook reads.
2. The file-format sections (FASTA, CSV/TSV, JSON) can be read independently once the demo files are created.
3. When running this notebook on your own data, remember to use absolute or relative paths that actually exist on your system.
4. The `pathlib` section shows the modern way to work with file paths — prefer it over string concatenation.

## Common stumbling points

- **Always use the `with` statement:** `with open(path) as f:` automatically closes the file when the block ends, even if an exception occurs. Forgetting to close files can cause data corruption (especially when writing) and resource leaks.
- **`r`, `w`, `a` modes:** Opening with `"w"` creates the file fresh — it **destroys existing content**. Use `"a"` to append. Use `"r"` (the default) to read.
- **`strip()` when reading lines:** Every line read from a file has a trailing `\n`. Always call `.strip()` on lines you read: `for line in f: line = line.strip()`. Forgetting this causes sequences to have invisible newlines that break comparisons.
- **Binary vs text mode:** Add `"b"` for binary files (e.g., BAM files): `open(path, "rb")`. Text mode (`"r"`) decodes bytes to strings using the system encoding — this fails or gives wrong results on binary files.
- **`FileNotFoundError`:** Check that the path exists and is spelled correctly (Python is case-sensitive on Linux/macOS). Use `pathlib.Path(path).exists()` to check before opening.
- **Reading the whole file into memory:** `f.read()` loads the entire file as a string. For large files (genomes, large FASTQ files), use `for line in f:` to process one line at a time.

[← Previous: Module 6: Functions](../06_Functions/01_functions.ipynb) | [Next: Module 8: Lists and Tuples →](../08_Lists_and_Tuples/01_lists_and_tuples.ipynb)

---

## 1. Basic File Operations

### File Modes

```
Mode   Description
----   -----------
'r'    Read (default) — file must exist
'w'    Write — creates file, overwrites if exists
'a'    Append — adds to end of existing file
'x'    Exclusive create — fails if file already exists
'r+'   Read and write
'rb'   Read in binary mode
'wb'   Write in binary mode
```

### Run this cell first to create demo files used throughout this notebook

In [ ]:
# Run this cell first — it creates the demo files used in this notebook
sample_fasta = """>seq1 Homo sapiens BRCA1
ATGGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
>seq2 Mus musculus Brca1 ortholog
ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
>seq3 Arabidopsis thaliana ATM kinase
ATGCCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
"""

with open('demo_sequences.fasta', 'w') as f:
    f.write(sample_fasta)

print("Demo files created:")
print("  demo_sequences.fasta (FASTA with 3 sequences)")
print("\nFirst 100 characters:")
print(sample_fasta[:100])

In [ ]:
# The OLD way (not recommended): manual open/close
f = open('demo_sequences.fasta', 'r')
print(f"File is open: {not f.closed}")
first_line = f.readline()
print(f"First line: {first_line.strip()}")
f.close()
print(f"File is closed: {f.closed}")
# Problem: if an error occurs before f.close(), the file stays open!

In [ ]:
# The RECOMMENDED way: context manager (with statement)
# The file is automatically closed when leaving the 'with' block,
# even if an exception occurs.

with open('demo_sequences.fasta', 'r') as f:
    content = f.read()
    print(f"Inside 'with': file open = {not f.closed}")

print(f"Outside 'with': file open = {not f.closed}")

---
## 2. Reading Methods

| Method | Description | Memory |
|--------|-------------|--------|
| `f.read()` | Entire file as one string | Loads all |
| `f.read(n)` | Read n characters | Partial |
| `f.readline()` | One line at a time | One line |
| `f.readlines()` | All lines into a list | Loads all |
| `for line in f:` | Iterate line by line | One line |

In [ ]:
# read() -- entire file as one string
with open('demo_sequences.fasta', 'r') as f:
    content = f.read()
print(f"read(): {len(content)} characters")
print(content[:80], "...")

In [ ]:
# readline() -- one line at a time
with open('demo_sequences.fasta', 'r') as f:
    line1 = f.readline()
    line2 = f.readline()
    print(f"Line 1: {line1.strip()}")
    print(f"Line 2: {line2.strip()}")

In [ ]:
# readlines() -- all lines into a list
with open('demo_sequences.fasta', 'r') as f:
    lines = f.readlines()
print(f"readlines(): {len(lines)} lines")
for i, line in enumerate(lines):
    print(f"  {i}: {line.strip()}")

In [ ]:
# BEST PRACTICE: iterate directly over file object
# Memory-efficient -- only one line is in memory at a time

with open('demo_sequences.fasta', 'r') as f:
    for line_num, line in enumerate(f, 1):
        print(f"  Line {line_num}: {line.strip()}")

---
## 3. Writing Files

In [ ]:
# write() -- writes a string (you must add \n yourself)
sequences = ["ATGCGATCG", "GCTAGCTAG", "TTAACCGGTT"]

with open('output.txt', 'w') as f:
    for seq in sequences:
        f.write(seq + '\n')

# Verify
with open('output.txt', 'r') as f:
    print(f.read())

In [ ]:
# 'a' mode appends to the end of an existing file
with open('output.txt', 'a') as f:
    f.write('AAATTTCCC\n')  # Added at the end

with open('output.txt', 'r') as f:
    print("After appending:")
    print(f.read())

In [ ]:
# writelines() -- writes a list of strings (no newlines added)
lines = ["Gene\tLength\tGC\n", "BRCA1\t7088\t42.3\n", "TP53\t2512\t51.2\n"]

with open('genes.tsv', 'w') as f:
    f.writelines(lines)

with open('genes.tsv', 'r') as f:
    print(f.read())

---
## 4. Text vs. Binary Files

- **Text mode** (`'r'`, `'w'`): reads/writes strings, handles line endings
- **Binary mode** (`'rb'`, `'wb'`): reads/writes raw bytes, needed for images, compressed files, serialized data

In [ ]:
# Writing and reading binary data
data = b'\x00\x01\x02\x03ATGC'  # bytes literal

with open('binary_demo.bin', 'wb') as f:
    f.write(data)

with open('binary_demo.bin', 'rb') as f:
    raw = f.read()

print(f"Raw bytes: {raw}")
print(f"Type: {type(raw)}")
print(f"Decoded text portion: {raw[4:].decode('ascii')}")

import os
os.remove('binary_demo.bin')

---
## 5. FASTA File Format

FASTA is the most common sequence format in bioinformatics.

```
>sequence_id description text
ATGCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGA
>another_sequence more info
GGGGCCCCAAAATTTT
```

Rules:
- Header lines start with `>`
- Sequence can span multiple lines (typically 60-80 chars per line)
- Empty lines are allowed and should be skipped

In [ ]:
# Create a more realistic FASTA file
fasta_content = """>gene1|BRCA1|Homo_sapiens DNA repair protein
ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATTAACCGGTTAACCGGTTAACCGGTTAACCGGTTAACCGG
TTAACCGGTTAA
>gene2|TP53|Homo_sapiens Tumor protein p53
GGGGCCCCAAAATTTTGGGGCCCCAAAATTTTGGGGCCCCAAAATTTTGGGGCCCCAAAATT
TTCCCCGGGGTTTTAAAACCCCGGGGTTTTAAAA
>gene3|EGFR|Homo_sapiens Epidermal growth factor receptor
ATATATATATATATATGCGCGCGCGCGCGCGCATATATATATATATATGCGCGCGCGCGCG
CGC
"""

with open('sequences.fasta', 'w') as f:
    f.write(fasta_content)

print("FASTA file created with 3 sequences.")

In [ ]:
def read_fasta(filename):
    """Parse a FASTA file and return a dictionary of sequences.
    
    Args:
        filename: Path to FASTA file
    
    Returns:
        dict mapping full headers to concatenated sequences
    """
    sequences = {}
    current_header = None
    current_seq = []
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:  # Skip empty lines
                continue
            
            if line.startswith('>'):
                # Save previous sequence if exists
                if current_header is not None:
                    sequences[current_header] = ''.join(current_seq)
                # Start new sequence
                current_header = line[1:]  # Remove '>'
                current_seq = []
            else:
                current_seq.append(line)
        
        # Don't forget the last sequence!
        if current_header is not None:
            sequences[current_header] = ''.join(current_seq)
    
    return sequences

# Test
seqs = read_fasta('sequences.fasta')
for header, seq in seqs.items():
    gc = (seq.count('G') + seq.count('C')) / len(seq) * 100
    print(f"{header.split()[0]}")
    print(f"  Length: {len(seq)} bp, GC: {gc:.1f}%")
    print(f"  First 40 bp: {seq[:40]}...")
    print()

In [ ]:
# Alternative FASTA parser using split (good for smaller files)
def read_fasta_split(filename):
    """Parse FASTA by splitting on '>' characters.
    
    Simpler but loads entire file into memory.
    
    Returns:
        list of (header, sequence) tuples
    """
    with open(filename) as f:
        entries = f.read().split('>')[1:]  # Split and drop empty first element
    
    result = []
    for entry in entries:
        lines = entry.strip().split('\n')
        header = lines[0]
        sequence = ''.join(lines[1:])
        result.append((header, sequence))
    
    return result

for header, seq in read_fasta_split('sequences.fasta'):
    print(f"{header.split('|')[1]}: {len(seq)} bp")

In [ ]:
def write_fasta(sequences, filename, line_width=60):
    """Write sequences to a FASTA file.
    
    Args:
        sequences: dict of {header: sequence} or list of (header, sequence)
        filename: Output file path
        line_width: Characters per sequence line (default: 60)
    """
    # Handle both dict and list input
    if isinstance(sequences, dict):
        items = sequences.items()
    else:
        items = sequences
    
    with open(filename, 'w') as f:
        for header, seq in items:
            f.write(f">{header}\n")
            # Write sequence in wrapped lines
            for i in range(0, len(seq), line_width):
                f.write(seq[i:i+line_width] + '\n')

# Test: filter GC-rich sequences and write to new file
gc_rich = {
    header: seq for header, seq in seqs.items()
    if (seq.count('G') + seq.count('C')) / len(seq) > 0.45
}

write_fasta(gc_rich, 'gc_rich.fasta')
print(f"Wrote {len(gc_rich)} GC-rich sequences to gc_rich.fasta")

# Verify
with open('gc_rich.fasta', 'r') as f:
    print(f.read())

---
## 6. CSV and TSV Files

Tabular data formats commonly used for gene expression data, annotations, and analysis results.

- **CSV** (Comma-Separated Values): fields separated by commas
- **TSV** (Tab-Separated Values): fields separated by tabs

In [ ]:
import csv

# Create sample gene expression CSV
expression_data = [
    ['gene_id', 'gene_name', 'length', 'gc_content', 'expression_sample1', 'expression_sample2'],
    ['ENSG0001', 'BRCA1', '7088', '42.3', '150.5', '175.2'],
    ['ENSG0002', 'TP53', '2512', '51.2', '89.3', '95.1'],
    ['ENSG0003', 'EGFR', '5616', '48.7', '245.8', '230.0'],
    ['ENSG0004', 'MYC', '2357', '55.1', '312.4', '298.7'],
]

# Write CSV
with open('gene_expression.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(expression_data)

print("CSV file created.")

In [ ]:
# Reading CSV with csv.reader
with open('gene_expression.csv', 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # First row is the header
    print(f"Columns: {header}")
    print()
    for row in reader:
        print(f"  {row[1]:6s} | Length: {row[2]:>5s} | GC: {row[3]}% | Expr: {row[4]}")

In [ ]:
# DictReader -- each row becomes a dictionary (much more readable code)
with open('gene_expression.csv', 'r') as f:
    reader = csv.DictReader(f)
    print("Highly expressed genes (>200 in sample 1):")
    for row in reader:
        if float(row['expression_sample1']) > 200:
            print(f"  {row['gene_name']}: {row['expression_sample1']}")

In [ ]:
# Writing CSV with DictWriter
results = [
    {'gene': 'BRCA2', 'fold_change': 2.5, 'p_value': 0.001},
    {'gene': 'TP73', 'fold_change': -1.8, 'p_value': 0.05},
    {'gene': 'MDM2', 'fold_change': 3.1, 'p_value': 0.0001},
]

with open('de_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['gene', 'fold_change', 'p_value'])
    writer.writeheader()
    writer.writerows(results)

# Read back
with open('de_results.csv', 'r') as f:
    print(f.read())

In [ ]:
# TSV: just use delimiter='\t'
bed_data = [
    ['chromosome', 'start', 'end', 'gene'],
    ['chr17', '43044295', '43170245', 'BRCA1'],
    ['chr17', '7661779', '7687550', 'TP53'],
    ['chr7', '55019021', '55211628', 'EGFR'],
]

with open('genes_bed.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerows(bed_data)

# Read TSV
with open('genes_bed.tsv', 'r') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        print(f"{row['gene']}: {row['chromosome']}:{row['start']}-{row['end']}")

---
## 7. JSON Format

JavaScript Object Notation -- ideal for structured, hierarchical data like gene annotations, API responses, and configuration files.

JSON maps naturally to Python types:
- JSON object `{}` -> Python `dict`
- JSON array `[]` -> Python `list`
- JSON string/number/boolean -> Python `str`/`int`/`float`/`bool`

In [ ]:
import json

# Gene annotation data -- nested structure
gene_annotation = {
    'gene_id': 'ENSG00000012048',
    'symbol': 'BRCA1',
    'description': 'BRCA1 DNA repair associated',
    'location': {
        'chromosome': '17',
        'start': 43044295,
        'end': 43170245,
        'strand': '-'
    },
    'exons': [
        {'number': 1, 'start': 43044295, 'end': 43044589},
        {'number': 2, 'start': 43047644, 'end': 43047703},
        {'number': 3, 'start': 43049094, 'end': 43049196},
    ],
    'transcripts': ['ENST00000357654', 'ENST00000352993'],
    'is_protein_coding': True,
    'diseases': ['Breast cancer', 'Ovarian cancer']
}

# Write JSON to file (indent for readability)
with open('gene.json', 'w') as f:
    json.dump(gene_annotation, f, indent=2)

print("JSON written. File contents:")
with open('gene.json', 'r') as f:
    print(f.read())

In [ ]:
# Read JSON from file
with open('gene.json', 'r') as f:
    data = json.load(f)

print(f"Gene: {data['symbol']} ({data['description']})")
print(f"Location: chr{data['location']['chromosome']}:{data['location']['start']}-{data['location']['end']}")
print(f"Strand: {data['location']['strand']}")
print(f"Exons: {len(data['exons'])}")
print(f"Associated diseases: {', '.join(data['diseases'])}")

In [ ]:
# JSON string conversion (without files) -- useful for APIs
record = {'sequence': 'ATGCGATCG', 'organism': 'E. coli'}

# To JSON string
json_str = json.dumps(record)
print(f"JSON string: {json_str}")
print(f"Type: {type(json_str)}")

# From JSON string
parsed = json.loads(json_str)
print(f"Parsed back: {parsed}")
print(f"Type: {type(parsed)}")

---
## 8. Pickle Format

Binary serialization format for Python objects. Fast to read/write and preserves complex Python types (sets, tuples, custom objects).

**Advantages:** Fast, compact, preserves all Python types.

**Limitations:** Python-specific (cannot be read by other languages), not human-readable. **Security warning:** never unpickle data from untrusted sources -- it can execute arbitrary code.

In [ ]:
import pickle

# Save complex analysis results
analysis_results = {
    'sequences': ['ATGCGATCG', 'GCTAGCTAG', 'TTAACCGG'],
    'gc_contents': [55.6, 55.6, 50.0],
    'similarity_matrix': [[1.0, 0.33, 0.22], [0.33, 1.0, 0.44], [0.22, 0.44, 1.0]],
    'valid_indices': {0, 1, 2},  # set -- cannot be saved as JSON!
    'metadata': {'date': '2026-04-02', 'tool': 'custom_pipeline v1.0'}
}

# Write pickle (note: binary mode 'wb')
with open('results.pkl', 'wb') as f:
    pickle.dump(analysis_results, f)

# Read pickle (note: binary mode 'rb')
with open('results.pkl', 'rb') as f:
    loaded = pickle.load(f)

print(f"Sequences: {loaded['sequences']}")
print(f"Valid indices (set): {loaded['valid_indices']}")
print(f"Metadata: {loaded['metadata']}")

---
## 9. GenBank Format Basics

GenBank is a richly annotated sequence format used by NCBI. While full parsing is complex (use BioPython for production code), understanding the structure is important.

```
LOCUS       AB000263  368 bp  mRNA  linear  PRI 05-FEB-1999
DEFINITION  Homo sapiens mRNA for ...
ACCESSION   AB000263
FEATURES             Location/Qualifiers
     source          1..368
                     /organism="Homo sapiens"
     CDS             86..>368
                     /codon_start=1
                     /product="HEXA"
ORIGIN
        1 agatggccaa tgccattgta ...
//
```

In [ ]:
# Simple GenBank sequence extractor
# (For full parsing, use Bio.SeqIO from BioPython)

genbank_content = """LOCUS       EXAMPLE  120 bp  DNA  linear  PLN 01-JAN-2026
DEFINITION  Example gene for demonstration.
ACCESSION   EX000001
FEATURES             Location/Qualifiers
     source          1..120
                     /organism="Arabidopsis thaliana"
                     /mol_type="genomic DNA"
     gene            1..120
                     /gene="EXAMPLE1"
     CDS             1..120
                     /gene="EXAMPLE1"
                     /product="example protein"
ORIGIN
        1 atggccgata agcttgatcc aaatgctcga tttaaagcgg ctaacccttg gcagtgcata
       61 atgacccgag accaaggcgt aatgttaggg agctttcaag agagtttacc cgatcgatag
//
"""

with open('example.gbk', 'w') as f:
    f.write(genbank_content)

def extract_genbank_sequence(filename):
    """Extract the DNA sequence from a GenBank file.
    
    Reads the ORIGIN section and strips numbers/spaces.
    """
    in_origin = False
    sequence_parts = []
    accession = None
    definition = None
    
    with open(filename) as f:
        for line in f:
            if line.startswith('ACCESSION'):
                accession = line.split()[1]
            elif line.startswith('DEFINITION'):
                definition = line[12:].strip()
            elif line.startswith('ORIGIN'):
                in_origin = True
            elif line.startswith('//'):
                break
            elif in_origin:
                # Remove numbers and spaces, keep only letters
                cleaned = ''.join(c for c in line if c.isalpha())
                sequence_parts.append(cleaned)
    
    return {
        'accession': accession,
        'definition': definition,
        'sequence': ''.join(sequence_parts).upper()
    }

result = extract_genbank_sequence('example.gbk')
print(f"Accession:  {result['accession']}")
print(f"Definition: {result['definition']}")
print(f"Sequence ({len(result['sequence'])} bp): {result['sequence']}")

---
## 10. Working with Large Files

Genomic files can be gigabytes in size. Never load them entirely into memory.

```
BAD:   data = file.read()          # Loads entire file -- may crash!
GOOD:  for line in file:           # One line at a time
GOOD:  chunk = file.read(1048576)  # Read 1 MB at a time
BEST:  Use generators              # Lazy evaluation
```

In [ ]:
# Generator-based FASTA parser (memory efficient for huge files)
def parse_fasta_generator(filename):
    """Yield (header, sequence) tuples one at a time.
    
    Memory usage is constant regardless of file size.
    """
    current_header = None
    current_seq = []
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header is not None:
                    yield (current_header, ''.join(current_seq))
                current_header = line[1:]
                current_seq = []
            else:
                current_seq.append(line)
        
        if current_header is not None:
            yield (current_header, ''.join(current_seq))

# Usage: processes one sequence at a time
print("Streaming FASTA sequences:")
for header, seq in parse_fasta_generator('sequences.fasta'):
    gene = header.split('|')[1]
    gc = (seq.count('G') + seq.count('C')) / len(seq) * 100
    print(f"  {gene}: {len(seq)} bp, {gc:.1f}% GC")

In [ ]:
# Counting sequences in a large file without loading it
def count_fasta_sequences(filename):
    """Count sequences in a FASTA file using streaming."""
    count = 0
    total_length = 0
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('>'):
                count += 1
            else:
                total_length += len(line.strip())
    
    return count, total_length

n_seqs, total_bp = count_fasta_sequences('sequences.fasta')
print(f"File contains {n_seqs} sequences totaling {total_bp} bp")

In [ ]:
# Reading in chunks (for non-line-based processing)
def process_file_in_chunks(filename, chunk_size=1024*1024):
    """Read a file in chunks of chunk_size bytes.
    
    Useful for calculating statistics on very large files.
    """
    total_bytes = 0
    chunks_read = 0
    
    with open(filename, 'r') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            total_bytes += len(chunk)
            chunks_read += 1
    
    return total_bytes, chunks_read

total, chunks = process_file_in_chunks('sequences.fasta', chunk_size=100)
print(f"Read {total} bytes in {chunks} chunks")

---
## 11. Exception Handling for Files

Always handle potential errors when working with files.

```
Common exceptions:
  FileNotFoundError   -- File does not exist
  PermissionError     -- No read/write permission
  IsADirectoryError   -- Tried to open directory as file
  IOError             -- General I/O problem
  UnicodeDecodeError  -- Encoding issue with text files
```

In [ ]:
# Basic try-except for file operations
try:
    with open('nonexistent_file.txt', 'r') as f:
        content = f.read()
except FileNotFoundError:
    print("Error: File not found!")

In [ ]:
# Robust FASTA reader with validation and error handling
class InvalidSequenceError(Exception):
    """Raised when a sequence contains invalid characters."""
    pass

def read_fasta_safe(filename, validate=True):
    """Read FASTA file with error handling and optional validation.
    
    Args:
        filename: Path to FASTA file
        validate: If True, check that sequences contain only ATGCN
    
    Returns:
        dict of {header: sequence}, or None on error
    """
    valid_chars = set('ATGCN')
    sequences = {}
    current_header = None
    current_seq = []
    
    try:
        with open(filename, 'r') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                
                if line.startswith('>'):
                    if current_header is not None:
                        sequences[current_header] = ''.join(current_seq)
                    current_header = line[1:]
                    current_seq = []
                else:
                    upper_line = line.upper()
                    if validate:
                        invalid = set(upper_line) - valid_chars
                        if invalid:
                            raise InvalidSequenceError(
                                f"Invalid characters {invalid} at line {line_num}"
                            )
                    current_seq.append(upper_line)
            
            if current_header is not None:
                sequences[current_header] = ''.join(current_seq)
    
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return None
    except InvalidSequenceError as e:
        print(f"Validation error: {e}")
        return None
    except PermissionError:
        print(f"Error: Permission denied for '{filename}'.")
        return None
    
    print(f"Successfully read {len(sequences)} sequences from {filename}")
    return sequences

# Test with valid file
result = read_fasta_safe('sequences.fasta')

# Test with nonexistent file
result = read_fasta_safe('does_not_exist.fasta')

---
## 12. File System Operations

Working with paths, directories, and finding files.

In [ ]:
from pathlib import Path
import os

# pathlib: the modern way to handle paths
p = Path('data/sequences/human/gene.fasta')
print(f"Name:      {p.name}")      # gene.fasta
print(f"Stem:      {p.stem}")      # gene
print(f"Suffix:    {p.suffix}")    # .fasta
print(f"Parent:    {p.parent}")    # data/sequences/human

# Build paths safely (works on all operating systems)
output = Path('results') / 'analysis' / 'output.csv'
print(f"\nBuilt path: {output}")

In [ ]:
# Find files with glob patterns
current = Path('.')

print("FASTA files in current directory:")
for f in sorted(current.glob('*.fasta')):
    size = f.stat().st_size
    print(f"  {f.name}: {size} bytes")

print("\nAll files:")
for f in sorted(current.glob('*.*')):
    if f.is_file():
        print(f"  {f.name}")

---
## 13. Complete Pipeline: FASTA Analysis to Report

Let us combine everything into a complete file-processing pipeline.

In [ ]:
import json
import csv
from pathlib import Path

def analyze_fasta_pipeline(input_fasta, output_dir='analysis_results'):
    """Complete pipeline: read FASTA, analyze, export CSV + JSON reports.
    
    Args:
        input_fasta: Path to input FASTA file
        output_dir: Directory for output files
    
    Returns:
        list of analysis result dicts
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # Step 1: Parse sequences
    sequences = read_fasta(input_fasta)
    
    # Step 2: Analyze each sequence
    results = []
    for header, seq in sequences.items():
        seq_id = header.split('|')[0].strip() if '|' in header else header.split()[0]
        gc = (seq.count('G') + seq.count('C')) / len(seq) * 100
        
        result = {
            'id': seq_id,
            'description': header,
            'length': len(seq),
            'gc_content': round(gc, 2),
            'a_count': seq.count('A'),
            't_count': seq.count('T'),
            'g_count': seq.count('G'),
            'c_count': seq.count('C'),
        }
        results.append(result)
    
    # Step 3: Write CSV summary
    csv_file = output_path / 'sequence_summary.csv'
    with open(csv_file, 'w', newline='') as f:
        fieldnames = ['id', 'length', 'gc_content', 'a_count', 't_count', 'g_count', 'c_count']
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    
    # Step 4: Write JSON with full details
    json_file = output_path / 'full_analysis.json'
    report = {
        'source': str(input_fasta),
        'total_sequences': len(results),
        'total_nucleotides': sum(r['length'] for r in results),
        'average_gc': round(sum(r['gc_content'] for r in results) / len(results), 2),
        'sequences': results
    }
    with open(json_file, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"Analysis complete!")
    print(f"  CSV report:  {csv_file}")
    print(f"  JSON report: {json_file}")
    
    return results

# Run the pipeline
results = analyze_fasta_pipeline('sequences.fasta')
print("\nSummary:")
for r in results:
    print(f"  {r['id']}: {r['length']} bp, {r['gc_content']}% GC")

In [ ]:
# Inspect the generated reports
print("=== CSV Report ===")
with open('analysis_results/sequence_summary.csv') as f:
    print(f.read())

print("\n=== JSON Report (first 500 chars) ===")
with open('analysis_results/full_analysis.json') as f:
    content = f.read()
    print(content[:500])

---
## 14. Practice Exercises

### Exercise 1: Complete FASTA Parser

Write a FASTA parser that returns a list of dictionaries, where each dict has 'id', 'description', and 'sequence' keys. The ID is the first word after `>`, and the description is the rest of the header line.

In [ ]:
# Exercise 1: Write your function here

def parse_fasta_records(filename):
    """Parse FASTA into a list of record dictionaries.
    
    Each record dict has:
        'id': First word after '>'
        'description': Everything after the ID on the header line
        'sequence': The concatenated sequence
    
    Args:
        filename: Path to FASTA file
    
    Returns:
        list of record dicts
    """
    # YOUR CODE HERE
    pass

# Test:
# records = parse_fasta_records('sequences.fasta')
# records[0] should have keys 'id', 'description', 'sequence'

In [ ]:
# Exercise 1: SOLUTION

def parse_fasta_records(filename):
    """Parse FASTA into a list of record dictionaries."""
    records = []
    current_id = None
    current_desc = ''
    current_seq = []
    
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_id is not None:
                    records.append({
                        'id': current_id,
                        'description': current_desc,
                        'sequence': ''.join(current_seq)
                    })
                parts = line[1:].split(None, 1)  # Split into max 2 parts
                current_id = parts[0]
                current_desc = parts[1] if len(parts) > 1 else ''
                current_seq = []
            else:
                current_seq.append(line)
        
        if current_id is not None:
            records.append({
                'id': current_id,
                'description': current_desc,
                'sequence': ''.join(current_seq)
            })
    
    return records

records = parse_fasta_records('sequences.fasta')
for r in records:
    print(f"ID: {r['id']}")
    print(f"  Description: {r['description']}")
    print(f"  Sequence length: {len(r['sequence'])} bp")
    print()

### Exercise 2: FASTQ Quality Score Reader

FASTQ format stores sequencing reads with quality scores:
```
@read_id
ATGCGATCG...
+
IIIIHHHHG...
```

Quality scores are ASCII characters. The Phred quality Q is: `Q = ord(char) - 33`.

Write a function that reads a FASTQ file and returns records with decoded quality scores.

In [ ]:
# Create a sample FASTQ file
fastq_content = """@read_001 length=30
ATGCGATCGATCGATCGATCGATCGATCGA
+
IIIIIIIIIHHHHHHHGGGGGGFFFFFEEE
@read_002 length=30
GCTAGCTAGCTAGCTAGCTAGCTAGCTAGC
+
IIIIIHHHHGGGGGFFFFF@@@@@555555
@read_003 length=30
TTAACCGGTTAACCGGTTAACCGGTTAACC
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIII
"""

with open('reads.fastq', 'w') as f:
    f.write(fastq_content)

print("FASTQ file created.")

In [ ]:
# Exercise 2: Write your function here

def read_fastq(filename):
    """Parse a FASTQ file.
    
    Args:
        filename: Path to FASTQ file
    
    Returns:
        List of dicts with 'id', 'sequence', 'quality_scores' (list of ints),
        and 'mean_quality' keys
    """
    # YOUR CODE HERE
    # Hint: FASTQ has 4-line records: @header, sequence, +, quality
    # Quality score: ord(char) - 33
    pass

# Test:
# reads = read_fastq('reads.fastq')
# for r in reads:
#     print(f"{r['id']}: mean quality = {r['mean_quality']:.1f}")

In [ ]:
# Exercise 2: SOLUTION

def read_fastq(filename):
    """Parse a FASTQ file."""
    records = []
    
    with open(filename) as f:
        while True:
            header = f.readline().strip()
            if not header:  # End of file
                break
            if not header.startswith('@'):
                continue
            
            sequence = f.readline().strip()
            f.readline()  # '+' line, skip it
            quality_str = f.readline().strip()
            
            quality_scores = [ord(c) - 33 for c in quality_str]
            mean_q = sum(quality_scores) / len(quality_scores) if quality_scores else 0
            
            records.append({
                'id': header[1:].split()[0],  # Remove @ and take first word
                'sequence': sequence,
                'quality_scores': quality_scores,
                'mean_quality': round(mean_q, 2)
            })
    
    return records

reads = read_fastq('reads.fastq')
for r in reads:
    min_q = min(r['quality_scores'])
    max_q = max(r['quality_scores'])
    print(f"{r['id']}: {len(r['sequence'])} bp, "
          f"quality min={min_q} max={max_q} mean={r['mean_quality']:.1f}")

### Exercise 3: CSV Expression Data Loader

Write a function that reads a gene expression CSV file and returns structured data for analysis. It should calculate fold changes between two samples.

In [ ]:
# Exercise 3: Write your function here

def load_expression_data(filename, sample1_col, sample2_col):
    """Load gene expression data and calculate fold changes.
    
    Args:
        filename: Path to CSV file
        sample1_col: Column name for sample 1 expression values
        sample2_col: Column name for sample 2 expression values
    
    Returns:
        List of dicts with gene info and fold_change (sample2/sample1)
    """
    # YOUR CODE HERE
    pass

# Test:
# data = load_expression_data('gene_expression.csv', 'expression_sample1', 'expression_sample2')
# for d in data:
#     print(f"{d['gene_name']}: fold_change = {d['fold_change']:.2f}")

In [ ]:
# Exercise 3: SOLUTION

import math

def load_expression_data(filename, sample1_col, sample2_col):
    """Load gene expression data and calculate fold changes."""
    results = []
    
    with open(filename, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            expr1 = float(row[sample1_col])
            expr2 = float(row[sample2_col])
            fold_change = expr2 / expr1 if expr1 > 0 else float('inf')
            
            results.append({
                'gene_id': row['gene_id'],
                'gene_name': row['gene_name'],
                'sample1_expression': expr1,
                'sample2_expression': expr2,
                'fold_change': round(fold_change, 4),
                'log2_fc': round(math.log2(fold_change), 4) if fold_change > 0 else None,
                'direction': 'up' if fold_change > 1 else ('down' if fold_change < 1 else 'unchanged')
            })
    
    return sorted(results, key=lambda x: abs(x['fold_change'] - 1), reverse=True)

# Test
data = load_expression_data('gene_expression.csv', 'expression_sample1', 'expression_sample2')
print(f"{'Gene':10s} {'Sample1':>10s} {'Sample2':>10s} {'FC':>8s} {'log2FC':>8s} {'Direction'}")
print('-' * 65)
for d in data:
    print(f"{d['gene_name']:10s} {d['sample1_expression']:10.1f} {d['sample2_expression']:10.1f} "
          f"{d['fold_change']:8.4f} {d['log2_fc']:8.4f} {d['direction']}")

### Exercise 4: Multi-FASTA Splitter

Write a function that splits a multi-sequence FASTA file into individual FASTA files, one per sequence.

In [ ]:
# Exercise 4: Write your function here

def split_fasta(input_file, output_dir='split_sequences'):
    """Split multi-FASTA into individual files.
    
    Each file is named after the sequence ID.
    
    Args:
        input_file: Path to multi-FASTA
        output_dir: Directory to create individual files in
    
    Returns:
        List of created file paths
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Exercise 4: SOLUTION

def split_fasta(input_file, output_dir='split_sequences'):
    """Split multi-FASTA into individual files."""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    created_files = []
    
    for header, seq in parse_fasta_generator(input_file):
        # Use first word as filename (replace problematic characters)
        seq_id = header.split()[0].replace('|', '_').replace('/', '_')
        out_file = output_path / f"{seq_id}.fasta"
        
        write_fasta({header: seq}, str(out_file))
        created_files.append(str(out_file))
    
    return created_files

# Test
files = split_fasta('sequences.fasta')
print(f"Created {len(files)} files:")
for f in files:
    print(f"  {f}")

### Exercise 5: Sequence Filter Pipeline

Write a function that reads a FASTA file, filters sequences by length and GC content, and writes the passing sequences to a new file.

In [ ]:
# Exercise 5: Write your function here

def filter_fasta(input_file, output_file, 
                 min_length=50, max_length=10000, 
                 min_gc=30.0, max_gc=70.0):
    """Filter FASTA sequences by length and GC content.
    
    Args:
        input_file: Input FASTA path
        output_file: Output FASTA path
        min_length: Minimum sequence length (inclusive)
        max_length: Maximum sequence length (inclusive)
        min_gc: Minimum GC content % (inclusive)
        max_gc: Maximum GC content % (inclusive)
    
    Returns:
        tuple of (total_count, passed_count)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Exercise 5: SOLUTION

def filter_fasta(input_file, output_file, 
                 min_length=50, max_length=10000, 
                 min_gc=30.0, max_gc=70.0):
    """Filter FASTA sequences by length and GC content."""
    total = 0
    passed = 0
    passed_seqs = {}
    
    for header, seq in parse_fasta_generator(input_file):
        total += 1
        length = len(seq)
        gc = (seq.count('G') + seq.count('C')) / length * 100 if length > 0 else 0
        
        if (min_length <= length <= max_length and 
            min_gc <= gc <= max_gc):
            passed_seqs[header] = seq
            passed += 1
    
    write_fasta(passed_seqs, output_file)
    print(f"Filtered: {passed}/{total} sequences passed (length {min_length}-{max_length}, GC {min_gc}-{max_gc}%)")
    return total, passed

# Test
total, passed = filter_fasta(
    'sequences.fasta', 'filtered.fasta',
    min_length=50, min_gc=40.0, max_gc=60.0
)

# Show filtered results
print("\nFiltered sequences:")
for header, seq in parse_fasta_generator('filtered.fasta'):
    gc = (seq.count('G') + seq.count('C')) / len(seq) * 100
    print(f"  {header.split('|')[1]}: {len(seq)} bp, {gc:.1f}% GC")

In [ ]:
# Cleanup temporary files
import shutil

temp_files = [
    'demo_sequences.fasta', 'output.txt', 'genes.tsv',
    'sequences.fasta', 'gc_rich.fasta', 'gene_expression.csv',
    'de_results.csv', 'genes_bed.tsv', 'gene.json', 'results.pkl',
    'example.gbk', 'reads.fastq', 'filtered.fasta'
]
temp_dirs = ['analysis_results', 'split_sequences']

for f in temp_files:
    if os.path.exists(f):
        os.remove(f)

for d in temp_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)

print("Temporary files cleaned up.")

---
## Summary

### Key Concepts

| Topic | Key Points |
|-------|------------|
| Context managers | Always use `with open()` -- automatic cleanup |
| Reading | `read()`, `readline()`, `readlines()`, or iterate |
| Writing | `write()`, `writelines()` with mode `'w'` or `'a'` |
| Text vs. binary | `'r'`/`'w'` for text, `'rb'`/`'wb'` for binary |
| FASTA | Header starts with `>`, sequence on following lines |
| CSV/TSV | Use `csv` module, `DictReader`/`DictWriter` for named columns |
| JSON | `json.dump()`/`json.load()` for files, `dumps()`/`loads()` for strings |
| Pickle | Binary serialization, Python-specific, never use with untrusted data |
| Large files | Use generators and line-by-line iteration |
| Exceptions | `try-except` for `FileNotFoundError`, `PermissionError`, etc. |
| Paths | Use `pathlib.Path` for modern, cross-platform path handling |

### Best Practices

1. Always use `with` statements for file operations
2. Strip whitespace from lines when parsing: `line.strip()`
3. Remember the last entry in parsers (the common "off-by-one" bug)
4. Use generators for large files (constant memory usage)
5. Validate data during parsing, not after
6. Handle file exceptions gracefully with informative messages

---

**Next module:** Data Structures -- lists, dictionaries, sets, and their applications in bioinformatics.

---[< Previous: Functions in Python](../06_Functions/01_functions.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Lists >](../08_Lists_and_Tuples/01_lists.ipynb)---